Readability analysis script
---------------------
This script analyzes an input .csv with LinkedIn scrapes for its readability according to 12 subcriteria, primarily derived from Shimamura et al. (2025). It's used for both ground truth sample and final LinkedIn dataset. Configuration for final readability analysis is the following:
- MODE = 1
- EXAMPLES = False
- REASONING = False
- SELF_CORRECTION = False
- ROLE_NR = 2
- MODEL = "GPT5.1"

In [1]:
import pandas as pd
from openai import OpenAI
import time
import json
import os
import sys
import textstat
from tqdm import tqdm
from dotenv import load_dotenv

configuration:

In [ ]:
# Load the API key from your specific .env file
load_dotenv(os.path.join("..", "API key.env"))
API_KEY = os.getenv("OPENAI_API_KEY")
if API_KEY: print("API_KEY loaded succesfully")

# Prompt Technique Toggles
EXAMPLES = False    # Set to True to include 3-shot examples input
REASONING = False   # Set to True for Chain-of-thought reasoning output
SELF_CORRECTION = False  # Set to True to make the LLM correct its previous output, only do this if the configuration already exists
ROLE_NR = 2                   # Choose the system prompt (role): "0" means leaving out the role in system prompt
ROLE = {
        0: "no role",
        1: "an expert in ESG investment",   # Shimamura et al., 2025    - readability
        2: "a professor of linguistics",   # Shimamura et al., 2025    - readability
        3: "an average worker",    # Shimamura et al., 2025    - readability
        4: "an ESG auditor",   # Jakubczak et al., 2025    - classification
        5: "a sustainability expert",  # Ghasemi, 2024     - classification
        6: "a trained research assistant"   # Tripp, 2024   - classification
         }[ROLE_NR]

# Model configuration
MODEL = "GPT5.1"
MODEL_ID = {"GPT5.1": "gpt-5.1", "GPT5": "gpt-5", "GPT5n": "gpt-5-nano"}[MODEL]  # This script works exclusively for OpenAI models
TEMPERATURE = 1     # 0 for strictly deterministic, however, GPT 5 only allows 1 (as it's a reasoning model)

# Files configuration
MODE = ["ground truth", "final dataset"][1] # Perform analysis on ground truth sample or final dataset
ROOT = "C:--FILL IN YOUR ROOT FOLDER--"  # root project folder
INPUT_DIR = {"final dataset": f"{ROOT}/Data Collection/LinkedIn", "ground truth": f"{ROOT}/Ground Truth/Samples"}[MODE]
TXT_FILES_DIR = "."
OUTPUT_DIR = {"final dataset": ".", "ground truth": "Ground Truth sample - rdbl/Readability analyses"}[MODE]

CATEGORY_FILE = "Readability criteria.txt"
OUTPUT_FILE = f"Readability analysis_{MODEL}_role{ROLE_NR}{"_examples" if EXAMPLES else ""}{"_reasoning" if REASONING else ""}.csv"
INPUT_FILE = {"final dataset": "LinkedIn 2025-26 Dataset - rdbl.csv", "ground truth": "LinkedIn Ground Truth Sample - rdbl.csv"}[MODE]
EXAMPLES_FILE = f"Readability analysis examples.txt"

# Script Parameters
MAX_POSTS = None                        # "None" for full analysis
POST_START = 0                       # Index of the post where analysis should start (0 is first post)
CHECKPOINT_INTERVAL = 500              # Save after x posts
SLEEP_TIME = 1                        # Break (in seconds) between API calls
RETRIES = 3                             # Amount of retries for 500 / 503 errors

# The 'edited' 13 Shimamura et al. (2025) readability criteria
SUBCRITERIA = [
    "A-01", "A-02", 
    "B-01", "B-02", "B-03", 
    "C-01", "C-02", "C-04", "C-05", "C-06", 
    "D-01",
    "L-01"
]

# Mapping to main criteria for calculating the averages
PARENT_MAP = {
    "A-01": "A", "A-02": "A",
    "B-01": "B", "B-02": "B", "B-03": "B",
    "C-01": "C", "C-02": "C", "C-04": "C", "C-05": "C", "C-06": "C",
    "D-01": "D",
    "L-01": "L"
}

Main algorithm:

In [ ]:
client = OpenAI(api_key=API_KEY)

def load_definitions(filepath):
    """Loads readability criteria from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Definition file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
    
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()

def load_examples(filepath):
    """
    Loads few-shot examples from text file. Raises error if file is missing. 
    Otherwise deletes reasoning in the examples if this configuration is disabled.
    """
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Examples file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
        
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        print(filepath, "opened successfully.")
        full_text =  f.read()

    # Opened file, moving on to filter out unwanted lines
    lines = full_text.splitlines()
    filtered_lines = []

    for line in lines:
        if not REASONING and '"reasoning":' in line:
            continue
        filtered_lines.append(line)

    # Clean up overhanging commas before closing brackets
    clean_lines = []
    total_lines = len(filtered_lines)
    for i in range(total_lines):
        current_line = filtered_lines[i]
        # Check if there is a next line, and if that next line contains a closing brace '}'
        if i + 1 < total_lines and "}" in filtered_lines[i + 1]:
            # If yes, the current line is the new final item; strip its trailing comma!
            current_line = current_line.rstrip(",")     
        clean_lines.append(current_line)
        
    # Rejoin all processed lines back into a single string
    return "\n".join(clean_lines)

def get_previous_output_string(row):
    """
    Reconstructs a readable JSON-like string from the dataframe row. 
    Only used in case of Self-Correction to integrate previous scores in prompt.
    """
    data = {}
    for sub in SUBCRITERIA:
        clean_sub = sub.replace("-", "_")

        score_val = row.get(f"{clean_sub}_score")
        reasoning_val = row.get(f"{clean_sub}_reasoning")
        
        # Setting nan to "N/A" string value
        if pd.isna(score_val) or score_val is None:
            score_val = "N/A"
        elif isinstance(score_val, float) and score_val.is_integer():
            score_val = int(score_val)  # converting floats to int
        
        sub_dict = {"score": score_val}
        if REASONING:
            sub_dict["reasoning"] = reasoning_val
            
        data[clean_sub] = sub_dict
    return json.dumps(data, indent=2)

def build_prompt(post_text, definitions, examples, pr_rdbl):
    """Dynamically builds the prompt based on configuration."""
    
    # Constructing the dynamic JSON schema description
    schema_fields = []
    for sub in SUBCRITERIA:
        clean_sub = sub.replace("-", "_")
        details = ['"score": "1", "2", "3", "4", "5", or "N/A"']
        if REASONING: details.append('"reasoning": "string"')
        schema_fields.append(f'"{clean_sub}": {{{", ".join(details)}}}')

    prompt = f"""{"""
Your task is to produce a verified analysis for the post below. 
You have already analyzed the LinkedIn post and are now reviewing your previous output.
1. Review your initial analysis critically. Ensure the scores (or N/A values) you granted reflect the text content.
2. If the initial analysis is correct, your verified output should match it exactly. If you find an error, your verified output must reflect the correction.
You must output the full JSON structure for every post.

Initial prompt:
""" if SELF_CORRECTION else ''}Your task is to analyze LinkedIn posts and score their readability using 13 criteria.

Definitions of the readability criteria:
{definitions}

Instructions:
1. Score the text for each of the criteria provided in the definitions
2. Use a 5-point Likert scale for all scores, definitions for scores 1, 3, and 5 are written out for each criterion. Use 2 or 4 for intermediate levels falling between those benchmarks
3. If the text physically lacks the element entirely (i.e., there are NO numbers, NO timeframes, or NO causal statements present in the text), you MUST output "N/A" for the score 
{ '4. Provide a detailed reasoning for every score or "N/A" value' if REASONING else '' }
The output should thus be a score of 1 to 5, or "N/A", for each criterion

{ f"Examples:\n{examples}" if EXAMPLES else ""}

Strict Output Format (JSON):
{{
    {", ".join(schema_fields)}
}}

Input (LinkedIn Post):
{post_text}{f"\n\nYour initial analysis:\n{pr_rdbl}" if SELF_CORRECTION else ""}"""
    
    return prompt

def get_response_schema():
    """Defines the JSON schema in OpenAI's expected format."""
    properties = {}
    
    # Define the structure for each subcriterion object and add each subcriterion to the schema dynamically
    for sub in SUBCRITERIA:
        clean_sub = sub.replace("-", "_")
        current_sub_props = {
            "score": {
                "type": "string", 
                "enum": ["1", "2", "3", "4", "5", "N/A"],
                "description": f"The Likert score from 1-5, or N/A for {sub}."
            }
        }

        if REASONING:
            current_sub_props["reasoning"] = {
                "type": "string",
                "description": f"Detailed justification for the score assigned to {sub}."
            }
        
        properties[clean_sub] = {
            "type": "object",
            "properties": current_sub_props,
            "required": list(current_sub_props.keys()),
            "additionalProperties": False
        }

    return {
        "type": "json_schema",
        "json_schema": {
            "name": "readability_analysis",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": properties,
                "required": list(properties.keys()),
                "additionalProperties": False
            }
        }
    }

def classify_post(post_text, definitions, examples, retries, pr_rdbl):
    """Sends prompt to GPT-5 using Structured Outputs."""
    prompt = build_prompt(post_text, definitions, examples, pr_rdbl)
    
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {ROLE}."},
                    {"role": "user", "content": prompt}
                ]  if ROLE_NR else [{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
                response_format=get_response_schema()
            )
            # OpenAI returns the string in .message.content
            return json.loads(response.choices[0].message.content)
            
        except Exception as e:
            if "503" in str(e) or "500" in str(e) or "rate_limit" in str(e).lower():
                print(f"\n[Attempt {attempt + 1}] API Error. Retrying...")
                time.sleep(10 * (attempt + 1))
                continue
            return {"error": str(e)}
            
    return {"error": "Maximum retries reached."}

def main():
    # Continue with the previous output if output has already been created for this configuration
    if os.path.exists(os.path.join(OUTPUT_DIR, OUTPUT_FILE)):
        input_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
        print("Continuing with previous output...")
    else:
        input_path = os.path.join(INPUT_DIR, INPUT_FILE)

    # Add "_selfc" to the output file if Self Correction is enabled
    output_file = OUTPUT_FILE.replace(".csv", "_selfc.csv") if SELF_CORRECTION else OUTPUT_FILE

    # 1. Verification of required files
    try:
        definitions = load_definitions(os.path.join(TXT_FILES_DIR, CATEGORY_FILE))
        examples = load_examples(os.path.join(TXT_FILES_DIR, EXAMPLES_FILE))
    except FileNotFoundError:
        sys.exit(1)

    if not os.path.exists(input_path):
        print(f"Error: Input file not found in {input_path}.")
        return

    # 2. Load Data
    df = pd.read_csv(input_path)
    print("Input file opened succesfully")
    # Expected columns: Company Name, Date, URL, Text
    text_col = "Text" 
    # If Self Correction is enabled: make use of original dataframe variable for reference scores
    df_orig = df.copy() if SELF_CORRECTION else None
    
    # 3. Initialize Columns
    # If Self Correction is enabled: wipe all existing columns
    # Initialize Gunning Fog column
    if "Gunning_Fog" not in df.columns: df["Gunning_Fog"] = None
    elif SELF_CORRECTION: df["Gunning_Fog"] = None

    # Initialize Subcriteria
    for sub in SUBCRITERIA:
        clean_sub = sub.replace("-", "_")
        cols = [f"{clean_sub}_score"]
        if REASONING: cols.append(f"{clean_sub}_reasoning")
        for col in cols:
            if col not in df.columns: df[col] = None
            elif SELF_CORRECTION: df[col] = None

    # Calculate range
    end_index = len(df)
    if MAX_POSTS is not None:
        end_index = min(POST_START + MAX_POSTS, len(df))

    print(f"Processing posts from index {POST_START} to {end_index}...")

    # 4. Processing Loop
    for index in tqdm(range(POST_START, end_index), desc="Analyzing Readability"):
        post_content = str(df.at[index, text_col])
        
         # Skip if already successfully processed
        if pd.notna(df.at[index, 'Gunning_Fog']) and "ERROR" not in str(df.at[index, 'Gunning_Fog']):
            continue
        
        # Get previous scores if Self Correction is enabled
        pr_rdbl = get_previous_output_string(df_orig.iloc[index]) if SELF_CORRECTION else None

        # 1. Calculate Gunning Fog (syntactic text metric)
        df.at[index, "Gunning_Fog"] = textstat.gunning_fog(post_content)

        # 2. Get LLM Analysis
        result = classify_post(post_content, definitions, examples, RETRIES, pr_rdbl)

        if "error" in result:
            df.at[index, 'Gunning_Fog'] = "ERROR: " + result["error"]
        else:
            # 3. Map Scores

            for sub in SUBCRITERIA:
                clean_sub = sub.replace("-", "_")
                sub_data = result.get(clean_sub, {})
                score_val = sub_data.get("score", "N/A")
                # Validation: check if score is in list of allowed values
                if str(score_val) not in ["1", "2", "3", "4", "5", "1.0", "2.0", "3.0", "4.0", "5.0", "N/A", None]:
                    df.at[index, 'Gunning_Fog'] = f"ERROR: Invalid LLM score format received ({score_val})"
                    break
                # Save individual score
                df.at[index, f"{clean_sub}_score"] = score_val
                if REASONING:
                    df.at[index, f"{clean_sub}_reasoning"] = sub_data.get("reasoning", "")

        # Save every X posts to prevent data loss
        if (index + 1) % CHECKPOINT_INTERVAL == 0:
            df.to_csv(os.path.join(OUTPUT_DIR, output_file), index=False, encoding='utf-8-sig')
            print(f" Checkpoint: Saved up to index {index}")

        time.sleep(SLEEP_TIME)

    # Final Save
    df.to_csv(os.path.join(OUTPUT_DIR, output_file), index=False, encoding='utf-8-sig')
    print(f"\nProcessing complete! Results saved to {output_file}")

if __name__ == "__main__":
    main()